# Advanced Machine Learning Workflow: Sensor Validation

**Core Requirements Implemented:**
- **Feature Extraction:** Trends (difference), Mean (rolling), Variance.
- **Models Evaluated:** Logistic Regression, Decision Tree, Random Forest, LSTM (Deep Learning).
- **Clinical Classifications:** Normal, Fever, Hypoxia, Abnormal/Critical.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

plt.style.use('ggplot')
sns.set_palette('viridis')

## 1. Data Processing and Feature Engineering

Extracting advanced features off the Hygeia Glove Data. The model learns not just from numeric values, but from how the patient is currently trending (Means, Variance, Difference over time).

In [ ]:
# 1. Load Data
df_raw = pd.read_csv('readings.csv', skiprows=2, header=None)

original_sensor = df_raw.iloc[:, [0, 1, 2]].apply(pd.to_numeric, errors='coerce')
original_sensor.columns = ['Orig_Temp', 'Orig_SpO2', 'Orig_HR']

glove_sensor = df_raw.iloc[:, [3, 4, 5]].apply(pd.to_numeric, errors='coerce')
glove_sensor.columns = ['Glove_Temp', 'Glove_SpO2', 'Glove_HR']

df_comb = pd.concat([original_sensor, glove_sensor], axis=1).dropna().reset_index(drop=True)

# 2. Classification Mapping (Original == Ground Truth)
def get_clinical_label(row):
    t, s, h = row['Orig_Temp'], row['Orig_SpO2'], row['Orig_HR']
    if s < 90 or t > 39 or h > 120 or h < 50: return 'Abnormal/ Critical'
    elif s < 92: return 'Hypoxia'
    elif t >= 38: return 'Fever'
    else: return 'Normal'
    
df_comb['Status'] = df_comb.apply(get_clinical_label, axis=1)

# 3. Trend, Mean, Variance Extraction
for feat in ['Glove_Temp', 'Glove_SpO2', 'Glove_HR']:
    df_comb[f"{feat}_mean"] = df_comb[feat].rolling(window=3, min_periods=1).mean()
    df_comb[f"{feat}_variance"] = df_comb[feat].rolling(window=3, min_periods=1).var().fillna(0)
    df_comb[f"{feat}_trend"] = df_comb[feat].diff().fillna(0)

# Keep only Glove specific features to validate prediction on device
glove_features = [col for col in df_comb.columns if 'Glove' in col]
X_raw = df_comb[glove_features]
y_raw = df_comb['Status']

print(f"Created {len(glove_features)} specialized features: {list(glove_features)}")
df_comb[glove_features].head()

## 2. Augmentation and Model Matrix Preparation

Expanding the dataset so models (especially LSTM) have sufficient material to evaluate. Encoding text targets into numeric representations.

In [ ]:
augmented_features = []
augmented_labels = []

for _ in range(20):
    noise = np.random.normal(0, 0.05, X_raw.shape)
    augmented_features.append(X_raw.values + noise)
    augmented_labels.append(y_raw.values)
    
X_massive = np.vstack(augmented_features)
y_massive = np.concatenate(augmented_labels)

le = LabelEncoder()
y_encoded = le.fit_transform(y_massive)

X_train, X_test, y_train, y_test = train_test_split(X_massive, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape} | Testing set shape: {X_test_scaled.shape}")

## 3. Four-Model Comparative Diagnostics

As defined in requirements, we test `Logistic Regression`, `Decision Tree`, `Random Forest`, and a Deep `LSTM` variant to identify the strongest classification performer.

In [ ]:
results = {}
trained_models = {}

# A. Logistic Regression (Baseline Model)
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)
results['LR'] = accuracy_score(y_test, lr.predict(X_test_scaled))
trained_models['LR'] = lr

# B. Decision Tree
dt = DecisionTreeClassifier(max_depth=6, random_state=42)
dt.fit(X_train_scaled, y_train)
results['DT'] = accuracy_score(y_test, dt.predict(X_test_scaled))
trained_models['DT'] = dt

# C. Random Forest (Ensemble Trees)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
results['RF'] = accuracy_score(y_test, rf.predict(X_test_scaled))
trained_models['RF'] = rf

# D. LSTM (Deep Pattern Detection)
X_train_lstm = np.expand_dims(X_train_scaled, axis=1) # Shape [Timesteps, Data]
X_test_lstm = np.expand_dims(X_test_scaled, axis=1)

lstm = Sequential([
    LSTM(64, input_shape=(1, X_train_scaled.shape[1])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(len(le.classes_), activation='softmax')
])
lstm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Training LSTM...")
lstm_history = lstm.fit(X_train_lstm, y_train, epochs=50, validation_data=(X_test_lstm, y_test), verbose=0)
results['LSTM'] = accuracy_score(y_test, np.argmax(lstm.predict(X_test_lstm, verbose=0), axis=1))
trained_models['LSTM'] = lstm

print("Training Complete for all 4 Classifiers!")

## 4. Evaluation Outputs and Conclusion Analytics

In [ ]:
# Determine Winner
best_model_name = max(results, key=results.get)
print(f"Highest Efficacy Observed In: {best_model_name}")

if best_model_name == 'LSTM':
    best_preds = np.argmax(trained_models['LSTM'].predict(X_test_lstm, verbose=0), axis=1)
else:
    best_preds = trained_models[best_model_name].predict(X_test_scaled)

print("\n=== FINAL CLASSIFICATION MATRIX ===")
print(classification_report(y_test, best_preds, target_names=le.classes_))

In [ ]:
plt.figure(figsize=(15, 6))

# Comparison Barchart
plt.subplot(1, 2, 1)
sns.barplot(x=list(results.keys()), y=list(results.values()), palette='magma')
plt.title('Performance Across the 4 Architectures')
plt.ylim(0, 1.1)
plt.ylabel('Test Accuracy Ratio')

# Ground Truth Correlation Matrix (Confusion Matrix)
plt.subplot(1, 2, 2)
cm = confusion_matrix(y_test, best_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f"Evaluation Matrices - {best_model_name}")
plt.xlabel('Diagnosed via ML (Glove Info)')
plt.ylabel('Actual Truth (Hospital Original)')

plt.tight_layout()
plt.show()